In [2]:
import sqlite3
import pandas as pd
import numpy as np

In [3]:
import sqlite3

conn = sqlite3.connect(
    r"C:\Users\adeeb\Downloads\archive (5)\travel.sqlite"
)

Q-Which routes have the highest passenger demand,
and how concentrated is total passenger traffic across the network?

In [4]:
route_fare_analysis = pd.read_sql_query("""
    SELECT
        CASE
            WHEN f.departure_airport < f.arrival_airport
            THEN f.departure_airport || ' ↔ ' || f.arrival_airport
            ELSE f.arrival_airport || ' ↔ ' || f.departure_airport
        END AS route,

        COUNT(tf.ticket_no) AS total_tickets,

        SUM(
            CASE
                WHEN tf.fare_conditions = 'Business'
                THEN 1
                ELSE 0
            END
        ) AS business_tickets,

        SUM(tf.amount) AS total_revenue,

        SUM(
            CASE
                WHEN tf.fare_conditions = 'Business'
                THEN tf.amount
                ELSE 0
            END
        ) AS business_revenue,

        AVG(tf.amount) AS average_revenue_per_ticket,
         AVG(
            CASE
                WHEN tf.fare_conditions = 'Business'
                THEN tf.amount
            END
        ) AS business_revenue_per_ticket

    FROM ticket_flights AS tf

    JOIN flights AS f
        ON tf.flight_id = f.flight_id

    GROUP BY
        CASE
            WHEN f.departure_airport < f.arrival_airport
            THEN f.departure_airport || ' ↔ ' || f.arrival_airport
            ELSE f.arrival_airport || ' ↔ ' || f.departure_airport
        END
""", conn)

route_fare_analysis


,route,total_tickets,business_tickets,total_revenue,business_revenue,average_revenue_per_ticket,business_revenue_per_ticket
0,AAQ ↔ EGO,8610,1073,68020200,20279700,7900.139373,18900.0
1,AAQ ↔ SVO,10626,1000,154633600,36600000,14552.380952,36600.0
2,ABA ↔ DME,1955,326,88060900,32926000,45043.938619,101000.0
3,ABA ↔ OVB,901,0,5225800,0,5800.000000,NaN
4,ABA ↔ TOF,948,0,4645200,0,4900.000000,NaN
...,...,...,...,...,...,...,...
224,UCT ↔ UFA,337,0,3336300,0,9900.000000,NaN
225,ULV ↔ VKO,4753,603,43014600,12964500,9049.989480,21500.0
226,UUD ↔ VKO,737,124,43826400,16516800,59465.943012,133200.0
227,VKO ↔ VOG,7101,888,80121900,23887200,11283.185467,26900.0


In [5]:
route_fare_analysis[
    ["route", "total_tickets"]
].sort_values(
    "total_tickets",
    ascending=False
)

,route,total_tickets
142,LED ↔ SVO,31885
209,SVO ↔ SVX,31312
68,DME ↔ OVB,31305
12,AER ↔ SVO,29816
183,OVB ↔ SVO,25997
...,...,...
137,KZN ↔ ROV,140
108,IKT ↔ MJZ,97
140,LED ↔ OVS,76
38,CEK ↔ SWT,44


In [6]:
top_10_passengers = (
    route_fare_analysis
    .sort_values("total_tickets", ascending=False)
    .head(10)["total_tickets"]
    .sum()
)

total_passengers = route_fare_analysis["total_tickets"].sum()

top_10_passenger_share = (
    top_10_passengers / total_passengers * 100
)

print(f"{top_10_passenger_share:.2f}% of total passengers travel on the top 10 routes.")

24.19% of total passengers travel on the top 10 routes.


In [7]:
top_20_passengers = (
    route_fare_analysis
    .sort_values("total_tickets", ascending=False)
    .head(20)["total_tickets"]
    .sum()
)

top_20_passenger_share = (
    top_20_passengers / total_passengers * 100
)

print(f"{top_20_passenger_share:.2f}% of total passengers travel on the top 20 routes.")

37.03% of total passengers travel on the top 20 routes.


229 × 25% = 57.25

In [8]:
top_25pct_passengers = (
    route_fare_analysis
    .sort_values("total_tickets", ascending=False)
    .head(57)["total_tickets"]
    .sum()
)

top_25pct_passenger_share = (
    top_25pct_passengers / total_passengers * 100
)

print(f"{top_25pct_passenger_share:.2f}% of total passengers travel on the top 25% of routes.")

66.74% of total passengers travel on the top 25% of routes.


The key finding

The busiest 25% of routes generate 66.74% of total passenger traffic.

Our business finding is:

Passenger demand is concentrated across the network: the top 25% of routes account for 66.74% of total passenger traffic, while the remaining 75% of routes account for only 33.26%. The top 10 routes alone contribute 24.19%.

Q5-Which departure and arrival airports handle the most passenger traffic, and how dependent is the network on its busiest airports?

In [9]:
#Get passenger traffic by departure airport
departure_traffic = pd.read_sql_query("""
    SELECT
        f.departure_airport,
        COUNT(tf.ticket_no) AS passenger_count
    FROM ticket_flights AS tf
    JOIN flights AS f
        ON tf.flight_id = f.flight_id
    GROUP BY f.departure_airport
""", conn)
departure_traffic

,departure_airport,passenger_count
0,AAQ,9502
1,ABA,1903
2,AER,32159
3,ARH,5128
4,ASF,2616
...,...,...
89,VKT,4258
90,VOG,15007
91,VOZ,1468
92,VVO,6006


In [10]:
#Get passenger traffic by arrival airport
arrival_traffic = pd.read_sql_query("""
    SELECT
        f.arrival_airport,
        COUNT(tf.ticket_no) AS passenger_count
    FROM ticket_flights AS tf
    JOIN flights AS f
        ON tf.flight_id = f.flight_id
    GROUP BY f.arrival_airport
    ORDER BY passenger_count DESC
""", conn)

arrival_traffic

,arrival_airport,passenger_count
0,SVO,150086
1,DME,139887
2,LED,66994
3,VKO,60040
4,OVB,47041
...,...,...
89,CEE,209
90,USK,201
91,NYA,132
92,RGK,92


In [11]:
#calculate each departure airport's share of total passenger traffic.
departure_traffic["passenger_share"] = (
    departure_traffic["passenger_count"]
    / departure_traffic["passenger_count"].sum()
)
departure_traffic = departure_traffic.sort_values(
    "passenger_count",
    ascending=False
)
top_10_departure_share = departure_traffic.head(10)["passenger_share"].sum()
top_10_departure_share



np.float64(0.5865360524649861)

In [12]:
arrival_traffic["passenger_share"] = (
    arrival_traffic["passenger_count"]
    / arrival_traffic["passenger_count"].sum()
)
arrival_traffic = arrival_traffic.sort_values(
    "passenger_count",
    ascending=False)
top_10_arrival_share = arrival_traffic.head(10)["passenger_share"].sum()
top_10_departure_share

np.float64(0.5865360524649861)

Q6. Which routes have high passenger demand but unusually low average fare, 
and which routes command premium fares despite relatively low demand?

In [13]:
route_fare_analysis[
    ["route", "total_tickets", "average_revenue_per_ticket"]
]

,route,total_tickets,average_revenue_per_ticket
0,AAQ ↔ EGO,8610,7900.139373
1,AAQ ↔ SVO,10626,14552.380952
2,ABA ↔ DME,1955,45043.938619
3,ABA ↔ OVB,901,5800.000000
4,ABA ↔ TOF,948,4900.000000
...,...,...,...
224,UCT ↔ UFA,337,9900.000000
225,ULV ↔ VKO,4753,9049.989480
226,UUD ↔ VKO,737,59465.943012
227,VKO ↔ VOG,7101,11283.185467


In [14]:
high_demand_threshold = route_fare_analysis["total_tickets"].quantile(0.75)
low_demand_threshold = route_fare_analysis["total_tickets"].quantile(0.25)

In [15]:
high_demand_threshold

np.float64(6423.0)

In [16]:
low_demand_threshold

np.float64(767.0)

In [17]:
low_fare_threshold = route_fare_analysis[
    "average_revenue_per_ticket"
].quantile(0.25)

high_fare_threshold = route_fare_analysis[
    "average_revenue_per_ticket"
].quantile(0.75)

In [18]:
low_fare_threshold

np.float64(8384.685659334054)

In [19]:
high_fare_threshold

np.float64(22780.65860775323)

In [20]:
high_demand_low_fare = route_fare_analysis[
    (route_fare_analysis["total_tickets"] >= high_demand_threshold) &
    (route_fare_analysis["average_revenue_per_ticket"] <= low_fare_threshold)
]

high_demand_low_fare = high_demand_low_fare.sort_values(
    "total_tickets",
    ascending=False
)

high_demand_low_fare[
    ["route", "total_tickets", "average_revenue_per_ticket"]
]

,route,total_tickets,average_revenue_per_ticket
142,LED ↔ SVO,31885,7994.329622
30,BZK ↔ SVO,10543,4639.466945
28,BZK ↔ DME,9349,4274.713873
170,NOZ ↔ OVB,9180,3763.572985
0,AAQ ↔ EGO,8610,7900.139373
78,DME ↔ ULV,8319,8384.685659
99,GOJ ↔ SVO,8196,4982.076623
222,TJM ↔ URJ,8031,4134.578508
188,PEE ↔ ULV,7991,7902.152421
129,KVX ↔ KZN,7951,4007.634260


In [21]:
low_demand_high_fare = route_fare_analysis[
    (route_fare_analysis["total_tickets"] <= low_demand_threshold) &
    (route_fare_analysis["average_revenue_per_ticket"] >= high_fare_threshold)
]

low_demand_high_fare = low_demand_high_fare.sort_values(
    "average_revenue_per_ticket",
    ascending=False
)

low_demand_high_fare[
    ["route", "total_tickets", "average_revenue_per_ticket"]
]

,route,total_tickets,average_revenue_per_ticket
92,GDX ↔ MRV,414,90033.574879
87,DYR ↔ SVO,546,80774.908425
93,GDX ↔ SCW,409,62092.420538
133,KXK ↔ SVX,767,61908.865711
226,UUD ↔ VKO,737,59465.943012
104,HTA ↔ UFA,721,50025.797503
127,KRR ↔ NOZ,186,42129.032258
10,AER ↔ NOZ,644,41726.708075
86,DYR ↔ KHV,572,40501.223776
152,MJZ ↔ SVX,11,40500.000000


The network contains distinct demand-pricing segments. Several high-volume routes, particularly LED–SVO, BZK–SVO and BZK–DME, fall into the network's high-demand/low-fare segment. Conversely, routes such as GDX–MRV, DYR–SVO and GDX–SCW combine relatively low passenger volumes with exceptionally high average fares. These differences warrant further investigation of fare-class mix and route economics.

Fare / Product Mix

How does the passenger mix across Economy and Business vary by route?

In [22]:
route_fare_mix = pd.read_sql_query("""
    SELECT
        CASE
            WHEN f.departure_airport < f.arrival_airport
            THEN f.departure_airport || ' ↔ ' || f.arrival_airport
            ELSE f.arrival_airport || ' ↔ ' || f.departure_airport
        END AS route,

        SUM(
            CASE
                WHEN tf.fare_conditions = 'Economy'
                THEN 1
                ELSE 0
            END
        ) AS economy_tickets,

        SUM(
            CASE
                WHEN tf.fare_conditions = 'Comfort'
                THEN 1
                ELSE 0
            END
        ) AS comfort_tickets,

        SUM(
            CASE
                WHEN tf.fare_conditions = 'Business'
                THEN 1
                ELSE 0
            END
        ) AS business_tickets,

        COUNT(tf.ticket_no) AS total_tickets

    FROM ticket_flights AS tf

    JOIN flights AS f
        ON tf.flight_id = f.flight_id

    GROUP BY
        CASE
            WHEN f.departure_airport < f.arrival_airport
            THEN f.departure_airport || ' ↔ ' || f.arrival_airport
            ELSE f.arrival_airport || ' ↔ ' || f.departure_airport
        END
""", conn)

route_fare_mix.head(10)

,route,economy_tickets,comfort_tickets,business_tickets,total_tickets
0,AAQ ↔ EGO,7537,0,1073,8610
1,AAQ ↔ SVO,9626,0,1000,10626
2,ABA ↔ DME,1629,0,326,1955
3,ABA ↔ OVB,901,0,0,901
4,ABA ↔ TOF,948,0,0,948
5,AER ↔ EGO,387,0,0,387
6,AER ↔ GOJ,410,0,0,410
7,AER ↔ KGP,189,0,23,212
8,AER ↔ KJA,920,0,200,1120
9,AER ↔ KUF,10543,0,1071,11614


In [23]:
route_fare_mix["economy_share"] = (
    route_fare_mix["economy_tickets"]
    / route_fare_mix["total_tickets"]
)

route_fare_mix["comfort_share"] = (
    route_fare_mix["comfort_tickets"]
    / route_fare_mix["total_tickets"]
)

route_fare_mix["business_share"] = (
    route_fare_mix["business_tickets"]
    / route_fare_mix["total_tickets"]
)
route_fare_mix["economy_share"] *= 100
route_fare_mix["comfort_share"] *= 100
route_fare_mix["business_share"] *= 100

In [24]:
route_fare_mix[
    [
        "route",
        "economy_share",
        "comfort_share",
        "business_share"
    ]
]


,route,economy_share,comfort_share,business_share
0,AAQ ↔ EGO,87.537747,0.0,12.462253
1,AAQ ↔ SVO,90.589121,0.0,9.410879
2,ABA ↔ DME,83.324808,0.0,16.675192
3,ABA ↔ OVB,100.000000,0.0,0.000000
4,ABA ↔ TOF,100.000000,0.0,0.000000
...,...,...,...,...
224,UCT ↔ UFA,100.000000,0.0,0.000000
225,ULV ↔ VKO,87.313276,0.0,12.686724
226,UUD ↔ VKO,83.175034,0.0,16.824966
227,VKO ↔ VOG,87.494719,0.0,12.505281


In [25]:
(
    route_fare_mix["economy_share"]
    + route_fare_mix["comfort_share"]
    + route_fare_mix["business_share"]
).head()

0    100.0
1    100.0
2    100.0
3    100.0
4    100.0
dtype: float64

In [26]:
route_fare_mix.sort_values(
    "business_share",
    ascending=False
).head(10)

,route,economy_tickets,comfort_tickets,business_tickets,total_tickets,economy_share,comfort_share,business_share
27,BTK ↔ DME,4411,0,986,5397,81.730591,0.0,18.269409
152,MJZ ↔ SVX,9,0,2,11,81.818182,0.0,18.181818
8,AER ↔ KJA,920,0,200,1120,82.142857,0.0,17.857143
123,KJA ↔ SVO,7660,0,1606,9266,82.667818,0.0,17.332182
180,OVB ↔ PEE,8309,0,1740,10049,82.684844,0.0,17.315156
147,LED ↔ YKS,1055,0,219,1274,82.810047,0.0,17.189953
106,IKT ↔ KZN,8118,0,1682,9800,82.836735,0.0,17.163265
104,HTA ↔ UFA,598,0,123,721,82.940361,0.0,17.059639
92,GDX ↔ MRV,344,0,70,414,83.091787,0.0,16.908213
226,UUD ↔ VKO,613,0,124,737,83.175034,0.0,16.824966


In [27]:
route_fare_mix.sort_values(
    "economy_share",
    ascending=False
).head(10)

,route,economy_tickets,comfort_tickets,business_tickets,total_tickets,economy_share,comfort_share,business_share
114,KGP ↔ NUX,654,0,0,654,100.0,0.0,0.0
105,IJK ↔ SVO,1442,0,0,1442,100.0,0.0,0.0
95,GDZ ↔ VKO,663,0,0,663,100.0,0.0,0.0
97,GOJ ↔ NAL,1536,0,0,1536,100.0,0.0,0.0
100,HMA ↔ NUX,624,0,0,624,100.0,0.0,0.0
102,HMA ↔ VKO,2152,0,0,2152,100.0,0.0,0.0
103,HTA ↔ NUX,2290,0,0,2290,100.0,0.0,0.0
182,OVB ↔ SLY,3881,0,0,3881,100.0,0.0,0.0
181,OVB ↔ SGC,515,0,0,515,100.0,0.0,0.0
70,DME ↔ PES,1380,0,0,1380,100.0,0.0,0.0


In [28]:
route_fare_mix.sort_values(
    "business_share",
    ascending=True
).head(10)

,route,economy_tickets,comfort_tickets,business_tickets,total_tickets,economy_share,comfort_share,business_share
114,KGP ↔ NUX,654,0,0,654,100.0,0.0,0.0
116,KHV ↔ UIK,800,0,0,800,100.0,0.0,0.0
181,OVB ↔ SGC,515,0,0,515,100.0,0.0,0.0
182,OVB ↔ SLY,3881,0,0,3881,100.0,0.0,0.0
112,KGD ↔ KRR,500,0,0,500,100.0,0.0,0.0
109,IKT ↔ VVO,860,0,0,860,100.0,0.0,0.0
108,IKT ↔ MJZ,97,0,0,97,100.0,0.0,0.0
186,OVS ↔ UFA,177,0,0,177,100.0,0.0,0.0
105,IJK ↔ SVO,1442,0,0,1442,100.0,0.0,0.0
103,HTA ↔ NUX,2290,0,0,2290,100.0,0.0,0.0


Passenger mix varies substantially by route. Some routes are exclusively Economy, while the most Business-oriented routes have roughly 17–18% of passengers traveling in Business. The data also shows that Comfort has no recorded passengers, making Economy vs Business the meaningful product-mix distinction in this dataset.

Q8 - Which routes have an unusually high share of Business passengers, and does this translate into higher revenue per passenger?

Combining the two DataFrames

We already have:

route_fare_mix → Business share
route_fare_analysis → average revenue per ticket

In [29]:
route_business_analysis = route_fare_mix[
    ["route", "business_share", "total_tickets"]
].merge(
    route_fare_analysis[
        ["route", "average_revenue_per_ticket"]
    ],
    on="route"
)

route_business_analysis

,route,business_share,total_tickets,average_revenue_per_ticket
0,AAQ ↔ EGO,12.462253,8610,7900.139373
1,AAQ ↔ SVO,9.410879,10626,14552.380952
2,ABA ↔ DME,16.675192,1955,45043.938619
3,ABA ↔ OVB,0.000000,901,5800.000000
4,ABA ↔ TOF,0.000000,948,4900.000000
...,...,...,...,...
224,UCT ↔ UFA,0.000000,337,9900.000000
225,ULV ↔ VKO,12.686724,4753,9049.989480
226,UUD ↔ VKO,16.824966,737,59465.943012
227,VKO ↔ VOG,12.505281,7101,11283.185467


In [30]:
# top 25% of routes
high_business_threshold = route_business_analysis[
    "business_share"
].quantile(0.75)

high_business_threshold

np.float64(12.483243967828418)

In [31]:
high_business_routes = route_business_analysis[
    route_business_analysis["business_share"] >= high_business_threshold
].sort_values(
    "business_share",
    ascending=False
)

high_business_routes[
    [
        "route",
        "total_tickets",
        "business_share",
        "average_revenue_per_ticket"
    ]
].head(10)

,route,total_tickets,business_share,average_revenue_per_ticket
27,BTK ↔ DME,5397,18.269409,52460.051881
152,MJZ ↔ SVX,11,18.181818,40500.000000
8,AER ↔ KJA,1120,17.857143,53176.160714
123,KJA ↔ SVO,9266,17.332182,44936.304770
180,OVB ↔ PEE,10049,17.315156,22385.610509
147,LED ↔ YKS,1274,17.189953,65218.602826
106,IKT ↔ KZN,9800,17.163265,47537.265306
104,HTA ↔ UFA,721,17.059639,50025.797503
92,GDX ↔ MRV,414,16.908213,90033.574879
226,UUD ↔ VKO,737,16.824966,59465.943012


In [32]:
#Calculate the correlation
route_business_analysis[
    ["business_share", "average_revenue_per_ticket"]
].corr()

,business_share,average_revenue_per_ticket
business_share,1.000000,0.469298
average_revenue_per_ticket,0.469298,1.000000


Business-heavy routes generally generate higher revenue per passenger, with a moderate positive correlation of 0.469 between Business passenger share and average revenue per ticket. However, the relationship is not strong enough to explain route revenue on its own, indicating that other factors such as route-specific pricing and demand also influence revenue.
    

Q9. Which fare classes contribute disproportionately to revenue relative to their passenger share?

In [33]:
fare_class_analysis = pd.read_sql_query("""
    SELECT
        tf.fare_conditions AS fare_class,
        COUNT(tf.ticket_no) AS total_tickets,
        SUM(tf.amount) AS total_revenue
    FROM ticket_flights AS tf
    GROUP BY tf.fare_conditions
""", conn)

fare_class_analysis

,fare_class,total_tickets,total_revenue
0,Business,107642,5505179600
1,Comfort,17291,566116900
2,Economy,920793,14695684400


In [34]:
#Passenger share
fare_class_analysis["passenger_share"] = (
    fare_class_analysis["total_tickets"]
    / fare_class_analysis["total_tickets"].sum()
) * 100

In [35]:
#Revenue share
fare_class_analysis["revenue_share"] = (
    fare_class_analysis["total_revenue"]
    / fare_class_analysis["total_revenue"].sum()
) * 100

In [36]:
fare_class_analysis

,fare_class,total_tickets,total_revenue,passenger_share,revenue_share
0,Business,107642,5505179600,10.293519,26.509292
1,Comfort,17291,566116900,1.653492,2.726043
2,Economy,920793,14695684400,88.052989,70.764665


In [37]:
fare_class_analysis["revenue_per_ticket"] = (
    fare_class_analysis["total_revenue"]
    / fare_class_analysis["total_tickets"]
)

fare_class_analysis


,fare_class,total_tickets,total_revenue,passenger_share,revenue_share,revenue_per_ticket
0,Business,107642,5505179600,10.293519,26.509292,51143.416139
1,Comfort,17291,566116900,1.653492,2.726043,32740.552889
2,Economy,920793,14695684400,88.052989,70.764665,15959.813335


Business is the airline's strongest revenue-generating fare class relative to its passenger volume. Although Business passengers account for only 10.3% of ticket-flight records, they contribute 26.5% of revenue and generate approximately 3.2× the revenue per ticket of Economy. Economy remains the primary volume driver, accounting for 88.1% of passengers and 70.8% of revenue.

Time / Booking

Q10. How does passenger demand vary over the observed period, and which periods show unusually high or low demand?

In [38]:
flights_df = pd.read_sql_query(
    "SELECT * FROM flights",
    conn
)

In [39]:
flights_df["scheduled_departure"] = (
    pd.to_datetime(flights_df["scheduled_departure"])
    .dt.tz_localize(None)
)

In [40]:
ticket_flights_df = pd.read_sql_query(
    "SELECT * FROM  ticket_flights",
    conn
)

In [41]:
flight_passengers = ticket_flights_df.merge(
    flights_df[["flight_id", "scheduled_departure"]],
    on="flight_id",
    how="inner"
)

In [42]:
flight_passengers["month"] = (
    flight_passengers["scheduled_departure"]
    .dt.to_period("M")
)

In [43]:
monthly_demand = (
    flight_passengers
    .groupby("month")
    .size()
    .reset_index(name="passenger_count")
)

In [44]:
monthly_demand

,month,passenger_count
0,2017-07,250854
1,2017-08,677976
2,2017-09,116896


Q11. How far in advance are passengers booking, and does booking lead time vary by fare class or route?

In [45]:
booking_lead_time = pd.read_sql_query("""
    SELECT
        t.ticket_no,
        t.book_ref,
        tf.flight_id,
        b.book_date,
        f.scheduled_departure,
        f.departure_airport,
        f.arrival_airport,
        tf.fare_conditions

    FROM tickets AS t

    JOIN bookings AS b
        ON t.book_ref = b.book_ref

    JOIN ticket_flights AS tf
        ON t.ticket_no = tf.ticket_no

    JOIN flights AS f
        ON tf.flight_id = f.flight_id
""", conn)

In [46]:
booking_lead_time["book_date"] = (
    pd.to_datetime(booking_lead_time["book_date"])
    .dt.tz_localize(None)
)

booking_lead_time["scheduled_departure"] = (
    pd.to_datetime(booking_lead_time["scheduled_departure"])
    .dt.tz_localize(None)
)

In [47]:
booking_lead_time["booking_lead_days"] = (
    booking_lead_time["scheduled_departure"]
    - booking_lead_time["book_date"]
).dt.total_seconds() / (24 * 60 * 60)

In [48]:
booking_lead_time["booking_lead_days"].describe()

count    1.045726e+06
mean     2.029498e+01
std      5.516409e+00
min      4.526389e+00
25%      1.583542e+01
50%      1.954306e+01
75%      2.469306e+01
max      5.027014e+01
Name: booking_lead_days, dtype: float64

Passengers book approximately 20 days in advance on average, with a median lead time of about 19.5 days. Most bookings fall roughly between 16 and 25 days before departure.

Does booking lead time vary by fare class?

In [49]:
booking_lead_time.groupby("fare_conditions")["booking_lead_days"].agg(
    ["count", "mean", "median"]
)

,count,mean,median
fare_conditions,,,
Business,107642,20.248806,19.464931
Comfort,17291,19.981953,19.194444
Economy,920793,20.306256,19.557639


Passengers book roughly 20 days before departure, and booking lead time is remarkably consistent across fare classes. Economy, Business, and Comfort passengers show very similar booking behavior, with median lead times of approximately 19–20 days.

In [50]:
booking_lead_time["route"] = np.where(
    booking_lead_time["departure_airport"] <
    booking_lead_time["arrival_airport"],

    booking_lead_time["departure_airport"] + " ↔ " +
    booking_lead_time["arrival_airport"],

    booking_lead_time["arrival_airport"] + " ↔ " +
    booking_lead_time["departure_airport"]
)

In [51]:
route_booking = (
    booking_lead_time
    .groupby("route")["booking_lead_days"]
    .agg(["count", "mean", "median"])
    .reset_index()
)
route_booking

,route,count,mean,median
0,AAQ ↔ EGO,8610,20.784930,20.121181
1,AAQ ↔ SVO,10626,19.758694,19.099653
2,ABA ↔ DME,1955,21.602387,20.885417
3,ABA ↔ OVB,901,20.329383,19.928472
4,ABA ↔ TOF,948,20.408633,20.150694
...,...,...,...,...
224,UCT ↔ UFA,337,19.575286,19.120833
225,ULV ↔ VKO,4753,20.872626,20.257639
226,UUD ↔ VKO,737,20.938164,20.415972
227,VKO ↔ VOG,7101,20.901912,20.161806


In [52]:
# 500 passenger records per route:
route_booking_500 = route_booking[
    route_booking["count"] >= 500
]

In [53]:
route_booking_500.sort_values(              # This identifies routes where passengers tend to book furthest ahead of departure.
    "median",
    ascending=False
).head(15)

,route,count,mean,median
86,DYR ↔ KHV,572,25.947835,25.112153
223,TOF ↔ VOG,589,25.114217,23.993056
87,DYR ↔ SVO,546,26.043599,23.592361
40,CNN ↔ DME,933,24.078483,22.750694
139,LED ↔ NOJ,519,22.269655,22.750000
116,KHV ↔ UIK,800,22.751516,21.982639
20,ASF ↔ BAX,981,22.952552,21.785417
35,CEK ↔ IKT,911,22.661138,21.757639
220,SVX ↔ UIK,800,22.755595,21.622569
169,NOJ ↔ VKO,1061,21.917639,21.511111


In [54]:
route_booking_500.sort_values(
    "median",
    ascending=True                   #This gives us the routes with the shortest booking lead time
).head(15)


,route,count,mean,median
155,MMK ↔ VKO,1804,19.143312,18.288194
59,DME ↔ LED,19382,19.591516,18.387847
142,LED ↔ SVO,31885,19.630479,18.612500
80,DME ↔ URS,2074,19.244207,18.687847
50,DME ↔ HMA,5660,19.714270,18.711458
175,OGZ ↔ SVO,3030,19.464128,18.745833
190,PES ↔ SVO,1043,19.437154,18.786806
63,DME ↔ NJC,3729,19.462143,18.815278
176,OGZ ↔ VKO,1767,19.762557,18.815972
52,DME ↔ KGD,7729,19.606842,18.848611


Passengers book approximately 20 days before departure overall, with a network median of 19.5 days. Booking behavior is almost identical across Economy, Business, and Comfort, indicating that fare class has little influence on booking timing. However, booking lead time varies more meaningfully by route. Some routes, such as DYR–KHV and TOF–VOG, have median booking lead times of approximately 24–25 days, while routes such as MMK–VKO and DME–LED are closer to 18–19 days. This suggests that booking behavior is more route-dependent than fare-class-dependent.
    

In [55]:
print("TEST")

TEST


In [56]:
print(conn)

In [57]:
pd.read_sql_query("SELECT * FROM flights LIMIT 1", conn)

,flight_id,flight_no,scheduled_departure,scheduled_arrival,departure_airport,arrival_airport,status,aircraft_code,actual_departure,actual_arrival
0,1185,PG0134,2017-09-10 09:50:00+03,2017-09-10 14:55:00+03,DME,BTK,Scheduled,319,\N,\N


In [58]:
route_demand = route_fare_analysis[
    ["route", "total_tickets"]
].copy()

route_demand = route_demand.rename(
    columns={"total_tickets": "total_passengers"}
)

route_demand.to_csv("route_demand.csv", index=False)

In [59]:
departure = departure_traffic.copy()
departure["airport_role"] = "Departure"

departure = departure.rename(
    columns={"passenger_count": "total_passengers"}
)

arrival = arrival_traffic.copy()
arrival["airport_role"] = "Arrival"

arrival = arrival.rename(
    columns={"passenger_count": "total_passengers"}
)

airport_demand = pd.concat(
    [departure, arrival],
    ignore_index=True
)

airport_demand.to_csv("airport_demand.csv", index=False)

In [60]:
monthly_demand.to_csv("monthly_demand.csv", index=False)

In [61]:
route_booking_500.to_csv("route_booking.csv", index=False)

In [62]:
airport_demand_clean = pd.concat([
    departure_traffic.rename(columns={
        "departure_airport": "airport",
        "passenger_count": "total_passengers"
    })[["airport", "total_passengers"]].assign(
        airport_role="Departure"
    ),

    arrival_traffic.rename(columns={
        "arrival_airport": "airport",
        "passenger_count": "total_passengers"
    })[["airport", "total_passengers"]].assign(
        airport_role="Arrival"
    )
], ignore_index=True)

airport_demand_clean.to_csv(
    "airport_demand_clean.csv",
    index=False
)

In [63]:
route_booking_powerbi = route_booking_500[
    ["route", "median"]
].copy()

route_booking_powerbi = route_booking_powerbi.rename(
    columns={"median": "median_booking_days"}
)

route_booking_powerbi.to_csv(
    "route_booking_powerbi.csv",
    index=False
)

In [64]:
route_pareto = route_demand[["route", "total_passengers"]].copy()

route_pareto = route_pareto.sort_values(
    "total_passengers",
    ascending=False
).reset_index(drop=True)

route_pareto["cumulative_passengers"] = (
    route_pareto["total_passengers"].cumsum()
)

route_pareto["cumulative_passenger_share"] = (
    route_pareto["cumulative_passengers"]
    / route_pareto["total_passengers"].sum()
)

route_pareto.to_csv(
    "route_pareto.csv",
    index=False
)

route_pareto.head(10)

,route,total_passengers,cumulative_passengers,cumulative_passenger_share
0,LED ↔ SVO,31885,31885,0.030491
1,SVO ↔ SVX,31312,63197,0.060434
2,DME ↔ OVB,31305,94502,0.090370
3,AER ↔ SVO,29816,124318,0.118882
4,OVB ↔ SVO,25997,150315,0.143742
5,PEE ↔ VKO,25946,176261,0.168554
6,LED ↔ VKO,21735,197996,0.189338
7,DME ↔ LED,19382,217378,0.207873
8,DME ↔ KHV,19013,236391,0.226054
9,DME ↔ KRR,16618,253009,0.241946


In [66]:
fare_class_analysis.to_csv("fare_class_analysis.csv", index=False)
route_fare_analysis.to_csv("route_fare_analysis.csv", index=False)
route_fare_mix.to_csv("route_fare_mix.csv", index=False)

In [67]:
high_demand_low_fare.to_csv("high_demand_low_fare.csv", index=False)
low_demand_high_fare.to_csv("low_demand_high_fare.csv", index=False)